In [8]:
import random
import tkinter as tk
from tkinter import messagebox, simpledialog

SUITS = ["Spades", "Hearts", "Diamonds", "Clubs"]
SYMBOLS = {"Spades": "♠", "Hearts": "♥", "Diamonds": "♦", "Clubs": "♣"}
RANKS = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]


def identify(card):
    return RANKS[card % 13], SUITS[card // 13]


def value(card):
    rank, _ = identify(card)
    if rank in ("J", "Q", "K"):
        return 10
    if rank == "A":
        return 11
    return int(rank)


def hand_value(hand):
    total = sum(value(c) for c in hand)
    aces = sum(1 for c in hand if c % 13 == 0)
    while total > 21 and aces:
        total -= 10
        aces -= 1
    return total


def blackjack(hand):
    return len(hand) == 2 and hand_value(hand) == 21


def can_split(hand):
    return len(hand) == 2 and hand[0] % 13 == hand[1] % 13

def set_button_visible(self, button, visible):
    if visible:
        button.pack(side="left", expand=True, padx=3, pady=5)
    else:
        button.pack_forget()

class Blackjack:
    def __init__(self, root):
        self.root = root
        root.title("Blackjack")
        root.geometry("800x700")
        root.configure(bg="#123D32")

        self.money = 1000
        self.highscore = 1000
        self.round_over = True
        self.phase = "betting"

        tk.Label(
            root, text="♠ BLACKJACK ♥",
            font=("Arial", 28, "bold"),
            bg="#123D32", fg="#F4D58D"
        ).pack(pady=15)

        self.score = tk.Label(
            root, font=("Arial", 14, "bold"),
            bg="#205143", fg="white", pady=10
        )
        self.score.pack(fill="x", padx=20)

        tk.Label(
            root, text="DEALER",
            font=("Arial", 14, "bold"),
            bg="#123D32", fg="white"
        ).pack(pady=(20, 5))

        self.dealer_display = tk.Frame(root, bg="#123D32")
        self.dealer_display.pack()

        self.dealer_total = tk.Label(
            root, text="", font=("Arial", 12),
            bg="#123D32", fg="white"
        )
        self.dealer_total.pack()

        tk.Label(
            root, text="YOUR HANDS",
            font=("Arial", 14, "bold"),
            bg="#123D32", fg="white"
        ).pack(pady=(20, 5))

        self.hands_display = tk.Frame(root, bg="#123D32")
        self.hands_display.pack(fill="x")

        self.status = tk.Label(
            root, text="Place your bet to begin!",
            font=("Arial", 14, "bold"),
            bg="#123D32", fg="#F4D58D",
            wraplength=700
        )
        self.status.pack(pady=15)

        betting = tk.Frame(root, bg="#123D32")
        betting.pack(pady=5)

        tk.Label(
            betting, text="Bet ($):",
            bg="#123D32", fg="white",
            font=("Arial", 12)
        ).pack(side="left", padx=5)

        self.bet_entry = tk.Entry(
            betting, width=10, font=("Arial", 12),
            justify="center"
        )
        self.bet_entry.insert(0, "50")
        self.bet_entry.pack(side="left", padx=5)

        self.deal_btn = tk.Button(
            betting,
            text="DEAL CARDS",
            command=self.start_round,
            font=("Arial", 12, "bold"),
            bg="#E8C879",
            fg="#173D31",
            activebackground="#F4D58D",
            width=14,
            padx=10,
            pady=8
        )
        self.deal_btn.pack(side="left", padx=8)

        controls = tk.Frame(root, bg="#123D32")
        controls.pack(side="top", fill="x", padx=10, pady=15)

        self.hit_btn = self.button(
            controls, "Hit", self.hit
        )
        self.stay_btn = self.button(
            controls, "Stay", self.stay
        )
        self.double_btn = self.button(
            controls, "Double", self.double_down
        )
        self.split_btn = self.button(
            controls, "Split", self.split
        )
        self.insurance_btn = self.button(
            controls, "Insurance", self.insurance
        )
        self.new_btn = self.button(
            controls, "New Round", self.new_round
        )

        # Display every button in the same row
        for btn in (
            self.hit_btn,
            self.stay_btn,
            self.double_btn,
            self.split_btn,
            self.insurance_btn,
            self.new_btn,
        ):
            btn.pack(side="left", expand=True, padx=3, pady=5)

        self.hit_btn = self.button(controls, "Hit", self.hit)
        self.stay_btn = self.button(controls, "Stay", self.stay)
        self.double_btn = self.button(controls, "Double", self.double_down)
        self.split_btn = self.button(controls, "Split", self.split)
        self.insurance_btn = self.button(controls, "Insurance", self.insurance)
        self.new_btn = self.button(controls, "New Round", self.new_round)

        self.update_ui()

    def button(self, parent, label, command):
        return tk.Button(
            parent, text=label, command=command,
            font=("Arial", 10, "bold"),
            bg="#E8C879", fg="#173D31",
            activebackground="#F4D58D",
            padx=8, pady=8, width=10
        )

    def draw(self):
        if not self.deck:
            self.deck = list(range(52))
            random.shuffle(self.deck)
        return self.deck.pop()

    def start_round(self):
        if not self.round_over:
            return

        try:
            bet = int(self.bet_entry.get())
        except ValueError:
            messagebox.showerror("Invalid bet", "Enter a whole number.")
            return

        if bet <= 0 or bet > self.money:
            messagebox.showerror(
                "Invalid bet", "Bet must be positive and no greater than your money."
            )
            return

        self.money -= bet
        self.bet = bet
        self.deck = list(range(52))
        random.shuffle(self.deck)
        self.draw()  # Burn card

        self.hands = [[self.draw(), self.draw()]]
        self.bets = [bet]
        self.doubles = [1]
        self.results = [None]
        self.current = 0
        self.dealer = [self.draw(), self.draw()]
        self.insurance_bet = 0
        self.revealed = False
        self.round_over = False
        self.phase = "playing"

        if blackjack(self.dealer):
            self.revealed = True
            self.results[0] = "push" if blackjack(self.hands[0]) else "lose"
            self.finish_round()
            return

        if self.dealer[1] % 13 == 0:
            self.phase = "insurance"
            self.status.config(text="Dealer shows an Ace. Take insurance?")
        elif blackjack(self.hands[0]):
            self.results[0] = "blackjack"
            self.finish_round()
        else:
            self.status.config(text="Your turn!")
        self.update_ui()

    def insurance(self):
        if self.phase != "insurance":
            return

        maximum = min(self.money, self.bets[0] // 2)
        amount = simpledialog.askinteger(
            "Insurance",
            f"Insurance bet (0 to decline, maximum ${maximum}):",
            parent=self.root, minvalue=0, maxvalue=maximum
        )
        if amount is None:
            return

        self.money -= amount
        self.insurance_bet = amount
        self.phase = "playing"

        if blackjack(self.hands[0]):
            self.results[0] = "blackjack"
            self.finish_round()
        else:
            self.status.config(text="Your turn!")
            self.update_ui()

    def hit(self):
        if self.phase != "playing":
            return
        self.hands[self.current].append(self.draw())

        total = hand_value(self.hands[self.current])
        if total > 21:
            self.results[self.current] = "lose"
            self.advance()
        elif total == 21:
            self.stay()
        else:
            self.status.config(text=f"You have {total}. Hit or Stay?")
            self.update_ui()

    def stay(self):
        if self.phase != "playing":
            return
        if self.results[self.current] is None:
            self.results[self.current] = "stand"
        self.advance()

    def double_down(self):
        if self.phase != "playing":
            return
        i = self.current

        if len(self.hands[i]) != 2:
            return
        if self.money < self.bets[i]:
            self.status.config(text="Not enough money to double.")
            return

        self.money -= self.bets[i]
        self.doubles[i] = 2
        self.hands[i].append(self.draw())

        if hand_value(self.hands[i]) > 21:
            self.results[i] = "lose"
        else:
            self.results[i] = "stand"
        self.advance()

    def split(self):
        if self.phase != "playing":
            return
        i = self.current
        hand = self.hands[i]

        if not can_split(hand):
            return
        if self.money < self.bets[i]:
            self.status.config(text="Not enough money to split.")
            return

        self.money -= self.bets[i]
        second_card = hand.pop()
        self.hands.insert(i + 1, [second_card])
        self.hands[i].append(self.draw())
        self.hands[i + 1].append(self.draw())
        self.bets.insert(i + 1, self.bets[i])
        self.doubles.insert(i + 1, 1)
        self.results.insert(i + 1, None)

        self.status.config(text="Split! Play your first hand.")
        self.update_ui()

    def advance(self):
        self.current += 1

        while self.current < len(self.hands):
            if self.results[self.current] is not None:
                self.current += 1
                continue
            if hand_value(self.hands[self.current]) >= 21:
                self.results[self.current] = "stand"
                self.current += 1
                continue
            break

        if self.current >= len(self.hands):
            self.play_dealer()
        else:
            self.status.config(
                text=f"Playing hand {self.current + 1} of {len(self.hands)}"
            )
            self.update_ui()

    def play_dealer(self):
        self.revealed = True

        if any(
            result != "lose" and hand_value(hand) <= 21
            for hand, result in zip(self.hands, self.results)
        ):
            while hand_value(self.dealer) < 17:
                self.dealer.append(self.draw())

        dealer_total = hand_value(self.dealer)

        for i, hand in enumerate(self.hands):
            if self.results[i] == "lose":
                continue

            player_total = hand_value(hand)

            # Player busts: player loses.
            if player_total > 21:
                self.results[i] = "lose"

            # Dealer busts: player wins.
            elif dealer_total > 21:
                self.results[i] = "win"

            # Compare totals when neither has busted.
            elif player_total > dealer_total:
                self.results[i] = "win"
            elif player_total < dealer_total:
                self.results[i] = "lose"
            else:
                self.results[i] = "push"

        self.finish_round()

    def finish_round(self):
        self.revealed = True

        for i, result in enumerate(self.results):
            stake = self.bets[i] * self.doubles[i]

            if result == "win":
                self.money += stake * 2
            elif result == "blackjack":
                self.money += stake + (3 * self.bets[i] // 2)
            elif result == "push":
                self.money += stake

        if self.insurance_bet and blackjack(self.dealer):
            self.money += self.insurance_bet * 3

        self.highscore = max(self.highscore, self.money)
        self.round_over = True
        self.phase = "finished"

        if any(r in ("win", "blackjack") for r in self.results):
            self.status.config(text="You won a hand! Start another round?")
        elif all(r == "push" for r in self.results):
            self.status.config(text="Push! Your bet has been returned.")
        else:
            self.status.config(text="Round over. Better luck next time!")

        self.update_ui()

    def new_round(self):
        if not self.round_over:
            return
    
        if self.money <= 0:
            if messagebox.askyesno(
                "Out of money", "Restart with $1,000?"
            ):
                self.money = 1000
            else:
                return
    
        self.phase = "betting"
        self.status.config(text="Place your bet to begin!")
        self.update_ui()

    def show_card(self, parent, card, hidden=False):
        if hidden:
            text, color, bg = "✦\nBLACKJACK", "#F4D58D", "#193E79"
        else:
            rank, suit = identify(card)
            text = f"{rank}\n\n{SYMBOLS[suit]}"
            color = "#C73535" if suit in ("Hearts", "Diamonds") else "#222222"
            bg = "#F9F6EE"

        tk.Label(
            parent, text=text, font=("Arial", 15, "bold"),
            width=5, height=3, bg=bg, fg=color,
            relief="raised", bd=3
        ).pack(side="left", padx=3, pady=3)

    def update_ui(self):
        self.score.config(
            text=f"Money: ${self.money}     "
                 f"Bet: ${getattr(self, 'bet', 0)}     "
                 f"High score: ${self.highscore}"
        )

        for widget in self.dealer_display.winfo_children():
            widget.destroy()

        for i, card in enumerate(getattr(self, "dealer", [])):
            self.show_card(
                self.dealer_display, card,
                hidden=(i == 0 and not self.revealed)
            )

        if getattr(self, "dealer", []):
            total = (
                f"Total: {hand_value(self.dealer)}"
                if self.revealed
                else f"Showing: {value(self.dealer[1])}"
            )
            self.dealer_total.config(text=total)
        else:
            self.dealer_total.config(text="")

        for widget in self.hands_display.winfo_children():
            widget.destroy()

        for i, hand in enumerate(getattr(self, "hands", [])):
            frame = tk.Frame(
                self.hands_display,
                bg="#205143" if i == getattr(self, "current", -1)
                and self.phase == "playing" else "#123D32",
                padx=8, pady=8
            )
            frame.pack(side="left", expand=True, padx=5)

            tk.Label(
                frame,
                text=f"Hand {i + 1} — Bet ${self.bets[i]}",
                bg=frame["bg"], fg="white",
                font=("Arial", 10, "bold")
            ).pack()

            cards = tk.Frame(frame, bg=frame["bg"])
            cards.pack()
            for card in hand:
                self.show_card(cards, card)

            tk.Label(
                frame, text=f"Total: {hand_value(hand)}",
                bg=frame["bg"], fg="#F4D58D",
                font=("Arial", 12, "bold")
            ).pack(pady=3)

            result = self.results[i]
            if result:
                tk.Label(
                    frame, text=result.upper(),
                    bg=frame["bg"], fg="white"
                ).pack()

            # Betting is available only between rounds.
            can_bet = self.round_over and self.money > 0
            
            self.bet_entry.config(
                state="normal" if can_bet else "disabled"
            )
            
            self.deal_btn.config(
                state="normal" if can_bet else "disabled"
            )

        playing = self.phase == "playing"
        self.hit_btn.config(state="normal" if playing else "disabled")
        self.stay_btn.config(state="normal" if playing else "disabled")
        self.insurance_btn.config(
            state="normal" if self.phase == "insurance" else "disabled"
        )
        self.new_btn.config(
            state="normal" if self.round_over else "disabled"
        )

        i = getattr(self, "current", 0)
        can_double = (
            playing and len(self.hands[i]) == 2
            and self.money >= self.bets[i]
        )
        can_split_now = (
            playing and can_split(self.hands[i])
            and self.money >= self.bets[i]
        )

        self.double_btn.config(
            state="normal" if can_double else "disabled"
        )
        self.split_btn.config(
            state="normal" if can_split_now else "disabled"
        )


if __name__ == "__main__":
    root = tk.Tk()
    game = Blackjack(root)
    root.mainloop()